## Decision Tree

A `Decision Tree` is a **supervised machine learning algorithm**
used for both **classification and regression**.
It makes decisions by asking a series of **if–else questions** on the data,
similar to how humans make decisions.

---

### What It Is

A Decision Tree:
- Splits data into smaller groups based on feature values
- Forms a tree-like structure of decisions
- Makes predictions by following a path from root to leaf
- Outputs a class (classification) or a value (regression)

---

### Why It Is Used

Decision Trees are used because:
- They are easy to understand and visualize
- They require little data preprocessing
- They work with both numerical and categorical data
- They can model non-linear relationships

---

### Where It Is Used

Decision Trees are commonly used in:
- Credit risk analysis
- Medical diagnosis
- Customer churn prediction
- Rule-based decision systems
- As base models in ensembles (Random Forest, Boosting)

---

### When It Performs Well

Decision Trees perform well when:
- Data has clear decision rules
- Relationships are non-linear
- Interpretability is important
- Dataset size is small to medium

---

### When It Performs Poorly

Decision Trees perform poorly when:
- Dataset is very large
- Data is noisy
- Tree grows too deep (overfitting)
- Small changes in data cause large changes in the tree

---

### Advantages

- Easy to interpret and explain
- Handles non-linear data well
- No need for feature scaling
- Works with mixed data types

---

### Limitations

- Prone to overfitting
- Sensitive to noise
- Unstable (small data changes → different tree)
- Lower accuracy compared to ensemble methods

---

### Key Takeaways

- Decision Trees mimic human decision-making
- Simple, interpretable, and flexible
- Can overfit if not controlled
- Foundation of Random Forest and Boosting


In [1]:
import numpy as np


### Gini Impurity

In [2]:
def gini(y):
    classes, counts = np.unique(y, return_counts=True)
    probs = counts / counts.sum()
    return 1 - np.sum(probs ** 2)


In [3]:
def split_dataset(X, y, feature_index, threshold):
    left_mask = X[:, feature_index] <= threshold
    right_mask = X[:, feature_index] > threshold
    
    return X[left_mask], y[left_mask], X[right_mask], y[right_mask]


### Find Best Split

In [4]:
def best_split(X, y):
    best_gini = float("inf")
    best_feature, best_threshold = None, None
    
    n_features = X.shape[1]
    
    for feature in range(n_features):
        thresholds = np.unique(X[:, feature])
        
        for threshold in thresholds:
            X_l, y_l, X_r, y_r = split_dataset(X, y, feature, threshold)
            
            if len(y_l) == 0 or len(y_r) == 0:
                continue
            
            weighted_gini = (
                len(y_l) / len(y) * gini(y_l) +
                len(y_r) / len(y) * gini(y_r)
            )
            
            if weighted_gini < best_gini:
                best_gini = weighted_gini
                best_feature = feature
                best_threshold = threshold
                
    return best_feature, best_threshold


### Tree Node Class


In [5]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value


### Build the Tree (Recursive)

In [7]:
def build_tree(X, y, depth=0, max_depth=5):
    num_samples, num_features = X.shape
    num_labels = len(np.unique(y))
    
    # stopping conditions
    if num_labels == 1 or num_samples == 0 or depth >= max_depth:
        leaf_value = np.bincount(y).argmax()
        return Node(value=leaf_value)
    
    feature, threshold = best_split(X, y)
    
    if feature is None:
        leaf_value = np.bincount(y).argmax()
        return Node(value=leaf_value)
    
    X_l, y_l, X_r, y_r = split_dataset(X, y, feature, threshold)
    
    left_child = build_tree(X_l, y_l, depth + 1, max_depth)
    right_child = build_tree(X_r, y_r, depth + 1, max_depth)
    
    return Node(feature, threshold, left_child, right_child)


### Prediction Logic

In [8]:
def predict_one(x, tree):
    if tree.value is not None:
        return tree.value
    
    if x[tree.feature] <= tree.threshold:
        return predict_one(x, tree.left)
    else:
        return predict_one(x, tree.right)


In [9]:
def predict(X, tree):
    return np.array([predict_one(x, tree) for x in X])


In [11]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(
    n_samples=200,
    n_features=2,
    n_classes=2,
    n_informative=2,
    n_redundant=0,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

tree = build_tree(X_train, y_train, max_depth=4)
preds = predict(X_test, tree)

accuracy = np.mean(preds == y_test)
print("Accuracy:", accuracy)


Accuracy: 0.725
